# JFDS Version 2: Popularity-Tier Gain + Maximum Genre Diversity — SVD

## Purpose


This notebook evaluates **Version 2** of the JFDS equation with a single base recommender. All methods receive the same unseen-item candidate pool.

It compares `TopK`, `Popularity`, `MMR`, `FairOnly`, and `JFDS`.

**Important correction:** `TopK` is the relevance-only baseline. The `JFDS` search space requires both the tier-balance and diversity terms to remain active, so it cannot be selected as an identical copy of TopK.


> **Run instructions:** Set `DATA_DIR` in the configuration cell to your MovieLens 1M directory, then use **Run All**.  
> This notebook uses chronological train/validation/test splits, freezes all selected settings before test evaluation, and writes to its own output folder.  
> For a smoke test, set `SEEDS = [42]`, `RUN_CANDIDATE_SENSITIVITY = False`, and optionally `TUNE_USER_LIMIT = 1200`. Restore the default full settings before final reporting.


## Version 2 equation

For user \(u\), candidate item \(i\), and selected list \(S\):

\[
\boxed{
JFDS(u,i,S)
=
\alpha\,RelNorm(u,i)
+
\beta\,TierGain(i,S)
+
\gamma\,DivGain(i,S)
}
\]

\[
\alpha,\beta,\gamma \ge 0,
\qquad
\alpha+\beta+\gamma=1.
\]

- \(RelNorm\) is the base-model score min–max normalized within the objective-only candidate pool.
- \(TierGain\) is the reduction in Jensen–Shannon divergence from the declared head/mid/tail target after adding \(i\).
- \(DivGain\) is \(1-\max_{j\in S} sim_{genre}(i,j)\), with zero diversity gain for an empty list.

The JFDS grid tests only non-degenerate configurations:

\[
\beta \ge 0.05,
\qquad
\gamma \ge 0.05.
\]

Validation still selects the best values of \(\alpha,\beta,\gamma\); this constraint only prevents JFDS from becoming the already-reported TopK baseline by setting both of its defining terms to zero.


In [ ]:
# ===========================
# 1. Imports and configuration
# ===========================
from __future__ import annotations

import json
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata, wilcoxon
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---- Data ----
DATA_DIR = Path("../dataset/movie_lens")  # change only if your MovieLens 1M folder is elsewhere
OUTPUT_DIR = Path("results_jfds_v2_svd_objective_only")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Reproducibility ----
SEEDS = [42, 52, 62]  # use [42] only for a smoke test; restore all three for final reporting
PRIMARY_SEED = SEEDS[0]


# ---- Chronological split ----
TRAIN_FRACTION = 0.72
VAL_FRACTION = 0.08
TEST_FRACTION = 0.20
MIN_INTERACTIONS_PER_USER = 10
MIN_TRAIN_INTERACTIONS_PER_USER = 3
POSITIVE_THRESHOLD = 4

# ---- Ranking ----
K = 10
PRIMARY_POOL_SIZE = 100
SENSITIVITY_POOL_SIZES = [50, 100, 200]
assert PRIMARY_POOL_SIZE in SENSITIVITY_POOL_SIZES


# ---- SVD training ----
MODEL_NAME = "SVD"
BATCH_SIZE = 4096
MODEL_PARAM_GRID = [
    {"n_factors": 20, "n_epochs": 12, "lr": 0.010, "reg": 0.020},
    {"n_factors": 40, "n_epochs": 12, "lr": 0.008, "reg": 0.025},
]


# ---- JFDS Version 2: popularity-tier gain + maximum genre-distance diversity ----
TIER_NAMES = np.array(["head", "mid", "tail"], dtype=object)
TARGET_TIER_SHARE = np.array([1/3, 1/3, 1/3], dtype=float)
SMOOTHING_PRIOR = 1.0

# ---- Validation-only tuning policy ----
# TopK remains the separate relevance-only baseline. Re-ranker weights are chosen
# only for the equation's fairness/diversity objective(s), never through an NDCG-loss gate.
TUNE_USER_LIMIT = None          # use every eligible validation user in final runs
WEIGHT_STEP = 0.05              # 0.05 simplex grid
MIN_ACTIVE_WEIGHT = 0.05        # every term used by a method must remain structurally active
SELECTION_POLICY = (
    "Validation-only, objective-only selection. Every non-degenerate candidate is considered; "
    "validation NDCG is logged as a descriptive outcome but is not an eligibility filter, "
    "Pareto objective, or tie-breaker."
)

# ---- Statistics and switches ----
N_BOOTSTRAP = 1000
RUN_BASE_MODEL_TUNING = True
RUN_CANDIDATE_SENSITIVITY = True

np.set_printoptions(suppress=True, precision=4)
print("Configuration loaded.")
print(f"DATA_DIR: {DATA_DIR.resolve()}")
print(f"Output:   {OUTPUT_DIR.resolve()}")


## 2. Load and preprocess MovieLens 1M

The expected files are `movies.dat`, `users.dat`, and `ratings.dat`. IDs are remapped only after loading so cold validation/test targets can be counted and reported.

In [ ]:
required_files = [DATA_DIR / name for name in ("movies.dat", "users.dat", "ratings.dat")]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "MovieLens 1M files were not found. Update DATA_DIR. Missing:\n" + "\n".join(missing)
    )

movies_raw = pd.read_csv(
    DATA_DIR / "movies.dat", sep="::", engine="python", encoding="latin-1", header=None,
    names=["MovieID", "Title", "Genres"],
)
users_df = pd.read_csv(
    DATA_DIR / "users.dat", sep="::", engine="python", encoding="latin-1", header=None,
    names=["UserID", "Gender", "Age", "Occupation", "Zip"],
)
ratings_raw = pd.read_csv(
    DATA_DIR / "ratings.dat", sep="::", engine="python", encoding="latin-1", header=None,
    names=["UserID", "MovieID", "Rating", "Timestamp"],
)

ratings_raw = ratings_raw.merge(movies_raw[["MovieID"]], on="MovieID", how="inner")
ratings_raw["_row_id"] = np.arange(len(ratings_raw), dtype=np.int64)
ratings_raw = ratings_raw.sort_values(["UserID", "Timestamp", "_row_id"], kind="mergesort").reset_index(drop=True)

print(f"movies: {len(movies_raw):,} | users: {len(users_df):,} | ratings: {len(ratings_raw):,}")
display(movies_raw.head())


## 3. Strict chronological train / validation / test split

Each retained user's oldest interactions are training data, the next interactions are validation data, and the newest interactions are test data.

In [ ]:
def chronological_three_way_split(
    ratings: pd.DataFrame,
    train_fraction: float = TRAIN_FRACTION,
    val_fraction: float = VAL_FRACTION,
    test_fraction: float = TEST_FRACTION,
    min_total: int = MIN_INTERACTIONS_PER_USER,
    min_train: int = MIN_TRAIN_INTERACTIONS_PER_USER,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Return train, validation, test, and a dropped-user report without temporal leakage."""
    if not math.isclose(train_fraction + val_fraction + test_fraction, 1.0, abs_tol=1e-9):
        raise ValueError("Train, validation, and test fractions must sum to 1.")

    train_parts, val_parts, test_parts, dropped_rows = [], [], [], []
    for user_id, group in ratings.groupby("UserID", sort=False):
        group = group.sort_values(["Timestamp", "_row_id"], kind="mergesort")
        n = len(group)
        if n < min_total:
            dropped_rows.append({"UserID": user_id, "n_interactions": n, "reason": "below_min_total"})
            continue

        n_test = max(1, int(np.ceil(n * test_fraction)))
        n_val = max(1, int(np.ceil(n * val_fraction)))
        n_train = n - n_val - n_test
        if n_train < min_train:
            dropped_rows.append({"UserID": user_id, "n_interactions": n, "reason": "below_min_train"})
            continue

        train_parts.append(group.iloc[:n_train])
        val_parts.append(group.iloc[n_train:n_train + n_val])
        test_parts.append(group.iloc[n_train + n_val:])

    train = pd.concat(train_parts, ignore_index=True)
    val = pd.concat(val_parts, ignore_index=True)
    test = pd.concat(test_parts, ignore_index=True)
    dropped = pd.DataFrame(dropped_rows)
    return train, val, test, dropped

train_df, val_df, test_df, dropped_users_df = chronological_three_way_split(ratings_raw)
trainval_df = pd.concat([train_df, val_df], ignore_index=True)

print(f"train: {len(train_df):,} | validation: {len(val_df):,} | test: {len(test_df):,}")
print(f"retained users: {train_df['UserID'].nunique():,} | dropped users: {len(dropped_users_df):,}")

# Split-disjointness assertions: row IDs are unique interaction identifiers.
train_ids, val_ids, test_ids = map(set, (train_df["_row_id"], val_df["_row_id"], test_df["_row_id"]))
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
assert len(train_ids | val_ids | test_ids) == len(train_df) + len(val_df) + len(test_df)

# Temporal-order assertions: ties are resolved using _row_id, so <= is allowed for timestamp ties.
for user_id in train_df["UserID"].unique():
    tr = train_df.loc[train_df.UserID == user_id, "Timestamp"]
    va = val_df.loc[val_df.UserID == user_id, "Timestamp"]
    te = test_df.loc[test_df.UserID == user_id, "Timestamp"]
    assert tr.max() <= va.min() <= te.min(), f"Temporal ordering failed for user {user_id}"

print("Split integrity checks passed.")


## 4. ID mapping, genres, and warm-start evaluation targets

Mappings are objective-only across the retained data. Genre vectors are L2-normalised multi-hot vectors.

In [ ]:
# Map all items/users retained in any partition. This permits transparent cold-target reporting.
retained_user_ids = np.sort(pd.concat([train_df.UserID, val_df.UserID, test_df.UserID]).unique())
retained_movie_ids = np.sort(pd.concat([train_df.MovieID, val_df.MovieID, test_df.MovieID]).unique())
user2idx = {user_id: idx for idx, user_id in enumerate(retained_user_ids)}
movie2idx = {movie_id: idx for idx, movie_id in enumerate(retained_movie_ids)}
idx2movie = {idx: movie_id for movie_id, idx in movie2idx.items()}
N_USERS, N_ITEMS = len(retained_user_ids), len(retained_movie_ids)

for frame in (train_df, val_df, test_df, trainval_df):
    frame["u_idx"] = frame["UserID"].map(user2idx).astype(int)
    frame["i_idx"] = frame["MovieID"].map(movie2idx).astype(int)

movies_df = movies_raw.set_index("MovieID").reindex(retained_movie_ids).reset_index()
all_genres = sorted({genre for genres in movies_df["Genres"] for genre in genres.split("|")})
genre2idx = {genre: idx for idx, genre in enumerate(all_genres)}

# L2-normalised multi-hot vectors make dot products equal cosine similarity.
genre_vec = np.zeros((N_ITEMS, len(all_genres)), dtype=np.float32)
for row in movies_df.itertuples(index=False):
    i = movie2idx[row.MovieID]
    for genre in row.Genres.split("|"):
        genre_vec[i, genre2idx[genre]] = 1.0
norms = np.linalg.norm(genre_vec, axis=1, keepdims=True)
genre_vec = genre_vec / np.maximum(norms, 1e-12)

users_df = users_df[users_df.UserID.isin(retained_user_ids)].copy()
users_df["u_idx"] = users_df["UserID"].map(user2idx).astype(int)


def build_seen_dict(frame: pd.DataFrame) -> dict[int, set[int]]:
    return frame.groupby("u_idx")["i_idx"].apply(set).to_dict()


def build_warm_targets(target_df: pd.DataFrame, fitted_item_mask: np.ndarray, label: str) -> tuple[dict, dict, list[int], pd.DataFrame]:
    """Build relevance/grade dictionaries only for target items supported by the fitting item universe."""
    target = target_df.copy()
    target["warm_item"] = fitted_item_mask[target["i_idx"].to_numpy()]
    target["is_relevant"] = target["Rating"] >= POSITIVE_THRESHOLD

    report = pd.DataFrame([{
        "split": label,
        "all_target_interactions": len(target),
        "cold_target_interactions": int((~target.warm_item).sum()),
        "cold_relevant_interactions": int((~target.warm_item & target.is_relevant).sum()),
        "users_with_any_cold_target": int(target.loc[~target.warm_item, "u_idx"].nunique()),
        "users_with_any_warm_relevant_target": int(target.loc[target.warm_item & target.is_relevant, "u_idx"].nunique()),
    }])

    warm_relevant = target[target.warm_item & target.is_relevant]
    relevant = warm_relevant.groupby("u_idx")["i_idx"].apply(set).to_dict()
    grades = warm_relevant.groupby("u_idx")[["i_idx", "Rating"]].apply(
        lambda group: dict(zip(group["i_idx"].astype(int), group["Rating"].astype(float)))
    ).to_dict()
    eval_users = sorted(relevant.keys())
    return relevant, grades, eval_users, report

print(f"N_USERS={N_USERS:,} | N_ITEMS={N_ITEMS:,} | N_GENRES={len(all_genres)}")


## 5. Training-only popularity tiers

Version 2 defines head, mid, and tail from interaction counts in the fitting split only. The target tier distribution is the declared policy target `[1/3, 1/3, 1/3]`; it is not a universal definition of fairness.


In [ ]:
def build_popularity_context(fit_df: pd.DataFrame, n_items: int) -> dict:
    pop_count = np.zeros(n_items, dtype=np.int64)
    counts = fit_df["i_idx"].value_counts()
    pop_count[counts.index.to_numpy(dtype=int)] = counts.to_numpy(dtype=np.int64)
    active_items = np.flatnonzero(pop_count > 0)
    if len(active_items) < 3:
        raise ValueError("At least three fitted items are required to create head/mid/tail tiers.")

    order = active_items[np.argsort(-pop_count[active_items], kind="stable")]
    n_active = len(order)
    head_end = max(1, int(np.ceil(0.20 * n_active)))
    mid_end = min(max(head_end + 1, int(np.ceil(0.50 * n_active))), n_active - 1)

    tier_codes = np.full(n_items, -1, dtype=np.int8)
    tier_codes[order[:head_end]] = 0
    tier_codes[order[head_end:mid_end]] = 1
    tier_codes[order[mid_end:]] = 2
    active_mask = pop_count > 0
    assert np.all(tier_codes[active_mask] >= 0)

    return {
        "pop_count": pop_count,
        "active_mask": active_mask,
        "active_items": active_items,
        "tier_codes": tier_codes,
        "tier_names": TIER_NAMES,
        "target": TARGET_TIER_SHARE.copy(),
    }

val_context = build_popularity_context(train_df, N_ITEMS)
test_context = build_popularity_context(trainval_df, N_ITEMS)

val_relevant, val_grades, val_eval_users, val_warm_report = build_warm_targets(
    val_df, val_context["active_mask"], "validation"
)
test_relevant, test_grades, test_eval_users, test_warm_report = build_warm_targets(
    test_df, test_context["active_mask"], "test"
)
warm_start_report = pd.concat([val_warm_report, test_warm_report], ignore_index=True)
display(warm_start_report)

train_seen = build_seen_dict(train_df)
trainval_seen = build_seen_dict(trainval_df)

print("Validation tier sizes:")
print(pd.Series(val_context["tier_codes"][val_context["active_mask"]]).map(dict(enumerate(TIER_NAMES))).value_counts())
print("Test tier sizes:")
print(pd.Series(test_context["tier_codes"][test_context["active_mask"]]).map(dict(enumerate(TIER_NAMES))).value_counts())


## 6. SVD base recommender

This notebook uses **biased matrix-factorisation SVD trained with explicit ratings**. It is fitted only on the supplied fitting split. Validation targets are never used in base-model fitting, and test targets are never used until the final frozen evaluation.


In [ ]:
def train_svd(
    fit_df: pd.DataFrame,
    n_users: int,
    n_items: int,
    params: dict,
    seed: int,
    batch_size: int = BATCH_SIZE,
    verbose: bool = True,
) -> dict:
    """Biased matrix factorisation for explicit ratings using mini-batch SGD."""
    rng = np.random.default_rng(seed)
    n_factors = int(params["n_factors"])
    n_epochs = int(params["n_epochs"])
    lr = float(params["lr"])
    reg = float(params["reg"])

    global_mean = float(fit_df["Rating"].mean())
    b_u = np.zeros(n_users, dtype=np.float64)
    b_i = np.zeros(n_items, dtype=np.float64)
    P = rng.normal(0.0, 0.05, size=(n_users, n_factors))
    Q = rng.normal(0.0, 0.05, size=(n_items, n_factors))

    users = fit_df["u_idx"].to_numpy(dtype=int)
    items = fit_df["i_idx"].to_numpy(dtype=int)
    ratings = fit_df["Rating"].to_numpy(dtype=float)

    for epoch in range(n_epochs):
        for start in range(0, len(users), batch_size):
            # A different but deterministic shuffle for each epoch/batch.
            if start == 0:
                order = rng.permutation(len(users))
            idx = order[start:start + batch_size]
            u, i, r = users[idx], items[idx], ratings[idx]
            pu, qi = P[u].copy(), Q[i].copy()
            bu_old, bi_old = b_u[u].copy(), b_i[i].copy()

            pred = global_mean + bu_old + bi_old + np.einsum("ij,ij->i", pu, qi)
            err = r - pred

            grad_bu = err - reg * bu_old
            grad_bi = err - reg * bi_old
            grad_p = err[:, None] * qi - reg * pu
            grad_q = err[:, None] * pu - reg * qi

            np.add.at(b_u, u, lr * grad_bu)
            np.add.at(b_i, i, lr * grad_bi)
            np.add.at(P, u, lr * grad_p)
            np.add.at(Q, i, lr * grad_q)

        if verbose:
            pred_train = global_mean + b_u[users] + b_i[items] + np.einsum("ij,ij->i", P[users], Q[items])
            rmse = float(np.sqrt(np.mean((ratings - pred_train) ** 2)))
            print(f"SVD epoch {epoch + 1:>2}/{n_epochs}: train RMSE={rmse:.4f}")

    return {"name": "SVD", "global_mean": global_mean, "b_u": b_u, "b_i": b_i, "P": P, "Q": Q, "params": params}

def score_user(model: dict, u_idx: int) -> np.ndarray:
    return (
        model["global_mean"]
        + model["b_u"][u_idx]
        + model["b_i"]
        + model["Q"] @ model["P"][u_idx]
    )

TRAIN_MODEL = train_svd
print("SVD training utilities ready.")


## 7. Candidate generation and Version 2 re-rankers

Every method receives the same objective-only unseen-item candidate pool for a user. Version 2 uses normalized relevance, reduction in tier-distribution Jensen–Shannon divergence, and maximum genre-distance diversity gain.

`TopK` is the separate relevance-only baseline. The JFDS grid excludes `beta = gamma = 0`, so JFDS cannot silently turn itself into TopK.


In [ ]:
def build_candidate_pool(
    model: dict,
    u_idx: int,
    seen_items: set[int],
    allowed_items_mask: np.ndarray,
    pool_size: int,
) -> tuple[np.ndarray, np.ndarray]:
    scores = np.asarray(score_user(model, u_idx), dtype=float)
    scores = np.nan_to_num(scores, nan=-np.inf, neginf=-np.inf, posinf=np.finfo(float).max)
    scores[~allowed_items_mask] = -np.inf
    if seen_items:
        scores[np.fromiter(seen_items, dtype=int)] = -np.inf

    finite = np.flatnonzero(np.isfinite(scores))
    n_take = min(pool_size, len(finite))
    if n_take == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    top = finite[np.argpartition(scores[finite], -n_take)[-n_take:]]
    order = np.lexsort((top, -scores[top]))  # descending score, ascending item ID for ties
    candidate_ids = top[order].astype(int)
    return candidate_ids, scores[candidate_ids].astype(float)


def build_candidate_pools(
    model: dict,
    users: list[int],
    seen_dict: dict[int, set[int]],
    allowed_items_mask: np.ndarray,
    pool_size: int,
) -> dict[int, tuple[np.ndarray, np.ndarray]]:
    pools = {}
    for u_idx in users:
        ids, scores = build_candidate_pool(
            model, int(u_idx), seen_dict.get(int(u_idx), set()), allowed_items_mask, pool_size
        )
        if len(ids) >= K:
            pools[int(u_idx)] = (ids, scores)
    return pools


def minmax_normalize(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    lo, hi = float(values.min()), float(values.max())
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return np.zeros_like(values, dtype=float)
    return (values - lo) / (hi - lo)


def js_divergence(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / np.maximum(p.sum(), 1e-12)
    q = q / np.maximum(q.sum(), 1e-12)
    midpoint = 0.5 * (p + q)

    def kl(a: np.ndarray, b: np.ndarray) -> float:
        mask = a > 0
        return float(np.sum(a[mask] * np.log2(a[mask] / b[mask])))

    return 0.5 * kl(p, midpoint) + 0.5 * kl(q, midpoint)


def simple_weight_grid(method: str, step: float = WEIGHT_STEP) -> list[dict]:
    """Return structurally non-degenerate simplex configurations for each re-ranker."""
    levels = np.round(np.arange(0.0, 1.0 + 1e-12, step), 10)
    configs = []
    if method in ("MMR", "DiversityOnly"):
        for alpha in levels:
            gamma = round(1.0 - float(alpha), 10)
            if alpha >= MIN_ACTIVE_WEIGHT - 1e-12 and gamma >= MIN_ACTIVE_WEIGHT - 1e-12:
                configs.append({"alpha": float(alpha), "beta": 0.0, "gamma": float(gamma)})
    elif method == "FairOnly":
        for alpha in levels:
            beta = round(1.0 - float(alpha), 10)
            if alpha >= MIN_ACTIVE_WEIGHT - 1e-12 and beta >= MIN_ACTIVE_WEIGHT - 1e-12:
                configs.append({"alpha": float(alpha), "beta": float(beta), "gamma": 0.0})
    elif method == "JFDS":
        for alpha in levels:
            for beta in levels:
                gamma = round(1.0 - float(alpha) - float(beta), 10)
                if (
                    alpha >= MIN_ACTIVE_WEIGHT - 1e-12
                    and beta >= MIN_ACTIVE_WEIGHT - 1e-12
                    and gamma >= MIN_ACTIVE_WEIGHT - 1e-12
                ):
                    configs.append({"alpha": float(alpha), "beta": float(beta), "gamma": float(gamma)})
    else:
        raise ValueError(f"Unknown method: {method}")

    dedup = {tuple(sorted(config.items())): config for config in configs}
    configs = list(dedup.values())
    if not configs:
        raise ValueError("No structurally non-degenerate weight configurations were generated.")
    for config in configs:
        assert abs(sum(config.values()) - 1.0) < 1e-8
        if method in ("MMR", "DiversityOnly"):
            assert config["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12
            assert config["gamma"] >= MIN_ACTIVE_WEIGHT - 1e-12
            assert abs(config["beta"]) < 1e-12
        elif method == "FairOnly":
            assert config["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12
            assert config["beta"] >= MIN_ACTIVE_WEIGHT - 1e-12
            assert abs(config["gamma"]) < 1e-12
        else:
            assert config["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12
            assert config["beta"] >= MIN_ACTIVE_WEIGHT - 1e-12
            assert config["gamma"] >= MIN_ACTIVE_WEIGHT - 1e-12
    return configs


def rerank(
    candidate_ids: np.ndarray,
    candidate_scores: np.ndarray,
    method: str,
    context: dict,
    weights: dict | None = None,
    k: int = K,
) -> list[int]:
    candidate_ids = np.asarray(candidate_ids, dtype=int)
    candidate_scores = np.asarray(candidate_scores, dtype=float)
    if len(candidate_ids) < k:
        return candidate_ids.tolist()
    if method == "TopK":
        return candidate_ids[:k].tolist()
    if method == "Popularity":
        order = np.lexsort((candidate_ids, -context["pop_count"][candidate_ids]))
        return candidate_ids[order[:k]].tolist()
    if weights is None:
        raise ValueError(f"{method} requires weights.")

    alpha = float(weights["alpha"])
    beta = float(weights["beta"])
    gamma = float(weights["gamma"])
    assert min(alpha, beta, gamma) >= -1e-12
    assert abs(alpha + beta + gamma - 1.0) < 1e-8
    if method == "MMR":
        assert abs(beta) < 1e-12
    elif method == "FairOnly":
        assert abs(gamma) < 1e-12
    elif method != "JFDS":
        raise ValueError(f"Unknown method: {method}")

    relevance = minmax_normalize(candidate_scores)
    tier_codes = context["tier_codes"]
    target = context["target"]
    remaining = np.ones(len(candidate_ids), dtype=bool)
    selected: list[int] = []
    tier_counts = np.zeros(3, dtype=float)

    for _ in range(k):
        positions = np.flatnonzero(remaining)
        items = candidate_ids[positions]
        rel = relevance[positions]

        if selected:
            similarity = genre_vec[items] @ genre_vec[np.asarray(selected, dtype=int)].T
            div = np.clip(1.0 - similarity.max(axis=1), 0.0, 1.0)
        else:
            div = np.zeros(len(items), dtype=float)

        raw_tier_gains = tier_gain_by_code(tier_counts, target)
        candidate_codes = tier_codes[items]
        assert np.all(candidate_codes >= 0)
        tier_component = minmax_normalize(raw_tier_gains[candidate_codes])

        if method == "MMR":
            score = alpha * rel + gamma * div
        elif method == "FairOnly":
            score = alpha * rel + beta * tier_component
        else:
            score = alpha * rel + beta * tier_component + gamma * div

        chosen_pos = int(positions[int(np.argmax(score))])
        chosen = int(candidate_ids[chosen_pos])
        selected.append(chosen)
        tier_counts[int(tier_codes[chosen])] += 1
        remaining[chosen_pos] = False

    assert len(selected) == k and len(set(selected)) == k
    assert set(selected).issubset(set(candidate_ids))
    return selected


print("Candidate and Version 2 re-ranking utilities ready.")


## 8. Metrics

Accuracy uses held-out ratings at or above the positive threshold. Version 2 reports per-list and rank-discounted aggregate tier Jensen–Shannon divergence, genre diversity, novelty, coverage, and exposure concentration separately.


In [ ]:
def ranking_metrics(rec_list: list[int], relevant: set[int], grades: dict[int, float], k: int = K) -> dict:
    rec = list(rec_list[:k])
    if not relevant:
        return {"precision": np.nan, "recall": np.nan, "ndcg": np.nan, "hitrate": np.nan}

    hits = np.array([item in relevant for item in rec], dtype=bool)
    precision = float(hits.mean()) if rec else 0.0
    recall = float(hits.sum() / len(relevant))
    hitrate = float(hits.any())

    gains = np.array([2.0 ** grades[item] - 1.0 if item in grades else 0.0 for item in rec], dtype=float)
    discounts = 1.0 / np.log2(np.arange(2, len(rec) + 2))
    dcg = float(np.sum(gains * discounts))
    ideal_gains = np.array(
        sorted((2.0 ** grade - 1.0 for grade in grades.values()), reverse=True)[:k], dtype=float
    )
    idcg = float(np.sum(ideal_gains * (1.0 / np.log2(np.arange(2, len(ideal_gains) + 2)))))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return {"precision": precision, "recall": recall, "ndcg": ndcg, "hitrate": hitrate}


def intra_list_diversity(rec_list: list[int]) -> float:
    items = np.asarray(rec_list, dtype=int)
    if len(items) < 2:
        return 0.0
    sim = genre_vec[items] @ genre_vec[items].T
    upper = sim[np.triu_indices(len(items), k=1)]
    return float(1.0 - upper.mean())


def mean_self_information(rec_list: list[int], context: dict) -> float:
    pop = context["pop_count"][np.asarray(rec_list, dtype=int)]
    active_count = max(1, int(context["active_mask"].sum()))
    probability = (pop + 1.0) / (context["pop_count"].sum() + active_count)
    return float(np.mean(-np.log2(probability)))


def mean_inverse_popularity(rec_list: list[int], context: dict) -> float:
    pop = context["pop_count"][np.asarray(rec_list, dtype=int)]
    return float(np.mean(1.0 / (pop + 1.0)))


def gini(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or np.allclose(values.sum(), 0.0):
        return 0.0
    values = np.sort(np.clip(values, 0.0, None))
    n = len(values)
    return float((2.0 * np.sum(np.arange(1, n + 1) * values) / (n * values.sum())) - (n + 1) / n)


def list_tier_divergence(rec_list: list[int], context: dict) -> float:
    codes = context["tier_codes"][np.asarray(rec_list, dtype=int)]
    counts = np.bincount(codes, minlength=3).astype(float)
    return js_divergence(counts / np.maximum(counts.sum(), 1e-12), context["target"])


def per_user_metric_row(
    u_idx: int,
    rec_list: list[int],
    relevant: dict[int, set[int]],
    grades: dict[int, dict[int, float]],
    context: dict,
) -> dict:
    row = ranking_metrics(rec_list, relevant[u_idx], grades[u_idx])
    row.update({
        "user": int(u_idx),
        "ild": intra_list_diversity(rec_list),
        "list_tier_js": list_tier_divergence(rec_list, context),
        "self_information": mean_self_information(rec_list, context),
        "inverse_popularity": mean_inverse_popularity(rec_list, context),
    })
    return row


def aggregate_exposure_metrics(rec_lists: dict[int, list[int]], context: dict, k: int = K) -> dict:
    exposure = np.zeros(len(context["pop_count"]), dtype=float)
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    for rec in rec_lists.values():
        items = np.asarray(rec[:k], dtype=int)
        np.add.at(exposure, items, discounts[:len(items)])

    tier_codes = context["tier_codes"]
    active = context["active_mask"]
    tail = tier_codes == 2
    recommended = exposure > 0
    tier_exposure = np.array([exposure[tier_codes == code].sum() for code in range(3)], dtype=float)
    tier_share = tier_exposure / max(tier_exposure.sum(), 1e-12)

    return {
        "catalogue_coverage": float(recommended[active].sum() / max(1, active.sum())),
        "tail_catalogue_coverage": float(recommended[tail].sum() / max(1, tail.sum())),
        "exposure_gini": gini(exposure[active]),
        "tier_exposure_js": js_divergence(tier_share, context["target"]),
        "tier_exposure_tv": float(0.5 * np.abs(tier_share - context["target"]).sum()),
        "head_exposure_share": float(tier_share[0]),
        "mid_exposure_share": float(tier_share[1]),
        "tail_exposure_share": float(tier_share[2]),
        "distinct_recommended_items": int(recommended[active].sum()),
    }


def evaluate_rec_lists(
    rec_lists: dict[int, list[int]],
    relevant: dict[int, set[int]],
    grades: dict[int, dict[int, float]],
    context: dict,
    seed: int,
    base_model: str,
    method: str,
) -> tuple[pd.DataFrame, dict]:
    rows = []
    for u_idx, rec in rec_lists.items():
        if u_idx in relevant:
            row = per_user_metric_row(u_idx, rec, relevant, grades, context)
            row.update({"seed": seed, "base_model": base_model, "method": method})
            rows.append(row)
    per_user = pd.DataFrame(rows)
    aggregate = aggregate_exposure_metrics(rec_lists, context)
    aggregate.update({"seed": seed, "base_model": base_model, "method": method, "n_eval_users": len(per_user)})
    return per_user, aggregate


## 9. Validation-only selection for SVD

Base-model hyperparameters are selected on `train_df → val_df`. Re-ranker weights are then selected only from validation candidate pools. The test partition remains untouched until all model parameters and JFDS weights are frozen.

The selector uses every structurally non-degenerate validation setting and selects a Pareto-efficient balance of lower tier divergence and maximum-distance genre diversity. Validation NDCG is logged for interpretation, but it never filters candidates, enters the Pareto objectives, or breaks ties.


In [ ]:
def choose_tuning_users(users: list[int], limit: int | None, seed: int) -> list[int]:
    users = np.asarray(sorted(users), dtype=int)
    if limit is None or len(users) <= limit:
        return users.tolist()
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(users, size=limit, replace=False).tolist())


def topk_validation_ndcg(pools: dict, relevant: dict, grades: dict) -> float:
    values = [
        ranking_metrics(ids[:K].tolist(), relevant[u], grades[u])["ndcg"]
        for u, (ids, _) in pools.items() if u in relevant
    ]
    return float(np.nanmean(values)) if values else np.nan


def tune_base_model(
    param_grid: list[dict],
    train_data: pd.DataFrame,
    n_users: int,
    n_items: int,
    validation_users: list[int],
    seen_dict: dict[int, set[int]],
    context: dict,
    relevant: dict,
    grades: dict,
    seed: int,
) -> tuple[dict, dict, pd.DataFrame]:
    records = []
    best_model, best_params, best_score = None, None, -np.inf
    for params in param_grid:
        print(f"\nTuning {MODEL_NAME} with {params}")
        model = TRAIN_MODEL(train_data, n_users, n_items, params, seed=seed, verbose=False)
        pools = build_candidate_pools(
            model, validation_users, seen_dict, context["active_mask"], PRIMARY_POOL_SIZE
        )
        score = topk_validation_ndcg(pools, relevant, grades)
        records.append({
            "seed": int(seed),
            "base_model": MODEL_NAME,
            **params,
            "validation_ndcg": float(score),
            "n_valid_users": int(len(pools)),
        })
        print(f"  validation NDCG@{K}: {score:.5f} over {len(pools):,} users")
        if score > best_score:
            best_model, best_params, best_score = model, params.copy(), score
    return best_model, best_params, pd.DataFrame(records).sort_values(
        "validation_ndcg", ascending=False
    ).reset_index(drop=True)


def pareto_efficient_mask(
    frame: pd.DataFrame,
    benefit_cols: list[str],
    cost_cols: list[str],
) -> np.ndarray:
    """Return True for rows that are not dominated on all listed objectives."""
    if frame.empty:
        return np.zeros(0, dtype=bool)
    benefits = frame[benefit_cols].to_numpy(dtype=float) if benefit_cols else np.empty((len(frame), 0))
    costs = frame[cost_cols].to_numpy(dtype=float) if cost_cols else np.empty((len(frame), 0))
    efficient = np.ones(len(frame), dtype=bool)
    for i in range(len(frame)):
        better_or_equal = np.ones(len(frame), dtype=bool)
        strictly_better = np.zeros(len(frame), dtype=bool)
        if benefit_cols:
            better_or_equal &= np.all(benefits >= benefits[i], axis=1)
            strictly_better |= np.any(benefits > benefits[i], axis=1)
        if cost_cols:
            better_or_equal &= np.all(costs <= costs[i], axis=1)
            strictly_better |= np.any(costs < costs[i], axis=1)
        better_or_equal[i] = False
        if np.any(better_or_equal & strictly_better):
            efficient[i] = False
    return efficient


def unit_interval(values: pd.Series) -> pd.Series:
    values = values.astype(float)
    lo, hi = values.min(), values.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return pd.Series(np.zeros(len(values), dtype=float), index=values.index)
    return (values - lo) / (hi - lo)


def tune_reranker(
    method: str,
    pools: dict[int, tuple[np.ndarray, np.ndarray]],
    relevant: dict[int, set[int]],
    grades: dict[int, dict[int, float]],
    context: dict,
    users: list[int],
) -> tuple[dict, pd.DataFrame]:
    """Select weights using only the stated non-accuracy objectives on validation data."""
    active_users = [u for u in users if u in pools and u in relevant]
    if not active_users:
        raise ValueError("No validation user has both a candidate pool and a warm-start relevant target.")

    topk_rec_lists = {u: pools[u][0][:K].tolist() for u in active_users}
    topk_ndcg = float(np.mean([
        ranking_metrics(topk_rec_lists[u], relevant[u], grades[u])["ndcg"] for u in active_users
    ]))
    topk_ild = float(np.mean([intra_list_diversity(topk_rec_lists[u]) for u in active_users]))
    topk_tier_js = float(np.mean([list_tier_divergence(topk_rec_lists[u], context) for u in active_users]))

    records = []
    for weights in simple_weight_grid(method):
        ndcg_values, ild_values, tier_js_values = [], [], []
        for u in active_users:
            ids, scores = pools[u]
            rec = rerank(ids, scores, method, context, weights, K)
            ndcg_values.append(ranking_metrics(rec, relevant[u], grades[u])["ndcg"])
            ild_values.append(intra_list_diversity(rec))
            tier_js_values.append(list_tier_divergence(rec, context))
        records.append({
            **weights,
            "validation_ndcg": float(np.mean(ndcg_values)),
            "validation_ild": float(np.mean(ild_values)),
            "validation_list_tier_js": float(np.mean(tier_js_values)),
            "n_users": int(len(active_users)),
        })

    history = pd.DataFrame(records)
    history["topk_validation_ndcg"] = topk_ndcg
    history["topk_validation_ild"] = topk_ild
    history["topk_validation_list_tier_js"] = topk_tier_js
    history["selection_status"] = "objective_only_non_degenerate"
    history["is_tuning_candidate"] = True

    # Accuracy is deliberately absent from the Pareto objectives and the final sort.
    if method == "MMR":
        benefit_cols, cost_cols = ["validation_ild"], []
    elif method == "FairOnly":
        benefit_cols, cost_cols = [], ["validation_list_tier_js"]
    elif method == "JFDS":
        benefit_cols, cost_cols = ["validation_ild"], ["validation_list_tier_js"]
    else:
        raise ValueError(method)

    pool = history.copy()
    pool["is_pareto"] = pareto_efficient_mask(pool, benefit_cols, cost_cols)
    pareto = pool.loc[pool["is_pareto"]].copy()
    if pareto.empty:
        pareto = pool.copy()

    if method == "MMR":
        pareto["selection_utility"] = unit_interval(pareto["validation_ild"])
    elif method == "FairOnly":
        pareto["selection_utility"] = unit_interval(-pareto["validation_list_tier_js"])
    else:
        pareto["selection_utility"] = 0.5 * unit_interval(
            -pareto["validation_list_tier_js"]
        ) + 0.5 * unit_interval(pareto["validation_ild"])

    # Deterministic structural tie-break only; no accuracy value is used.
    chosen = pareto.sort_values(
        ["selection_utility", "alpha", "beta", "gamma"],
        ascending=[False, False, False, False],
    ).iloc[0]

    history["is_pareto"] = pool["is_pareto"].to_numpy()
    history["selection_utility"] = np.nan
    history.loc[pareto.index, "selection_utility"] = pareto["selection_utility"].to_numpy()
    history["selected"] = (
        np.isclose(history["alpha"], chosen["alpha"])
        & np.isclose(history["beta"], chosen["beta"])
        & np.isclose(history["gamma"], chosen["gamma"])
    )

    selected = {key: float(chosen[key]) for key in ("alpha", "beta", "gamma")}
    if method == "MMR":
        assert selected["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12
        assert selected["gamma"] >= MIN_ACTIVE_WEIGHT - 1e-12
    elif method == "FairOnly":
        assert selected["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12
        assert selected["beta"] >= MIN_ACTIVE_WEIGHT - 1e-12
    elif method == "JFDS":
        assert selected["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12
        assert selected["beta"] >= MIN_ACTIVE_WEIGHT - 1e-12
        assert selected["gamma"] >= MIN_ACTIVE_WEIGHT - 1e-12

    return selected, history.sort_values(
        ["selected", "is_pareto", "selection_utility", "alpha", "beta", "gamma"],
        ascending=[False, False, False, False, False, False],
    )


print(f"{MODEL_NAME} tuning utilities ready.")


## 10. Synthetic checks for the equation and selection grid


In [ ]:
# Every JFDS candidate must use relevance, fairness, and diversity together.
jfds_grid = simple_weight_grid("JFDS")
assert all(w["alpha"] >= MIN_ACTIVE_WEIGHT - 1e-12 for w in jfds_grid)
assert all(w["beta"] >= MIN_ACTIVE_WEIGHT - 1e-12 for w in jfds_grid)
assert all(w["gamma"] >= MIN_ACTIVE_WEIGHT - 1e-12 for w in jfds_grid)
assert not any(
    np.isclose(w["alpha"], 0.0) or np.isclose(w["beta"], 0.0) or np.isclose(w["gamma"], 0.0)
    for w in jfds_grid
)

assert not any(np.isclose(w["beta"], 0.0) and np.isclose(w["gamma"], 0.0) for w in jfds_grid)

# Empty-list tier gains are symmetric; after head-heavy selection, mid/tail gains improve.
empty_gains = tier_gain_by_code(np.zeros(3), TARGET_TIER_SHARE)
head_heavy_gains = tier_gain_by_code(np.array([5.0, 0.0, 0.0]), TARGET_TIER_SHARE)
assert np.allclose(empty_gains, empty_gains[0])
assert head_heavy_gains[1] > head_heavy_gains[0]
assert head_heavy_gains[2] > head_heavy_gains[0]
print(f"Version 2 checks passed: {len(jfds_grid)} structurally non-degenerate JFDS weight settings.")

## 11. Full SVD experiment runner

For each seed:

1. Select base-model parameters using train → validation only.
2. Select non-degenerate re-ranker weights from validation candidates only.
3. Refit the base model on train + validation.
4. Evaluate all methods once on the untouched test partition.


In [ ]:
def run_single_seed(seed: int) -> dict:
    start = time.time()
    print("\n" + "=" * 90)
    print(f"RUNNING {MODEL_NAME} SEED {seed}")
    print("=" * 90)

    tuning_users = choose_tuning_users(val_eval_users, TUNE_USER_LIMIT, seed)
    selected = {"seed": int(seed), "base_model": MODEL_NAME, "base_params": {}, "rerankers": {}}

    print(f"\n--- {MODEL_NAME}: base-model validation tuning ---")
    if RUN_BASE_MODEL_TUNING:
        val_model, chosen_params, base_history = tune_base_model(
            MODEL_PARAM_GRID, train_df, N_USERS, N_ITEMS, tuning_users,
            train_seen, val_context, val_relevant, val_grades, seed
        )
    else:
        chosen_params = MODEL_PARAM_GRID[0].copy()
        val_model = TRAIN_MODEL(train_df, N_USERS, N_ITEMS, chosen_params, seed=seed, verbose=False)
        base_history = pd.DataFrame([{
            "seed": int(seed), "base_model": MODEL_NAME, **chosen_params,
            "validation_ndcg": np.nan, "n_valid_users": 0,
        }])

    selected["base_params"] = chosen_params
    val_pools = build_candidate_pools(
        val_model, tuning_users, train_seen, val_context["active_mask"], PRIMARY_POOL_SIZE
    )

    print(f"--- {MODEL_NAME}: non-degenerate re-ranker tuning on validation only ---")
    reranker_histories = []
    for method in ("MMR", "FairOnly", "JFDS"):
        weights, history = tune_reranker(
            method, val_pools, val_relevant, val_grades, val_context, tuning_users
        )
        history.insert(0, "method", method)
        history.insert(0, "base_model", MODEL_NAME)
        history.insert(0, "seed", int(seed))
        reranker_histories.append(history)
        selected["rerankers"][method] = weights
        selected_row = history.loc[history["selected"]].iloc[0]
        print(
            f"  {method} selected: {weights} | "
            f"validation NDCG={selected_row['validation_ndcg']:.5f} (descriptive only) | "
            f"status={selected_row['selection_status']}"
        )

    print(f"--- {MODEL_NAME}: final refit on train + validation ---")
    final_model = TRAIN_MODEL(trainval_df, N_USERS, N_ITEMS, chosen_params, seed=seed, verbose=False)
    final_pools = build_candidate_pools(
        final_model, test_eval_users, trainval_seen, test_context["active_mask"], PRIMARY_POOL_SIZE
    )

    methods = ["TopK", "Popularity", "MMR", "FairOnly", "JFDS"]
    all_per_user, all_aggregate = [], []
    for method in methods:
        rec_lists = {}
        for u_idx, (candidate_ids, candidate_scores) in final_pools.items():
            weights = None if method in ("TopK", "Popularity") else selected["rerankers"][method]
            rec = rerank(candidate_ids, candidate_scores, method, test_context, weights, K)
            assert len(rec) == K and len(set(rec)) == K
            assert set(rec).issubset(set(candidate_ids))
            assert not (set(rec) & trainval_seen.get(u_idx, set()))
            rec_lists[u_idx] = rec
        per_user, aggregate = evaluate_rec_lists(
            rec_lists, test_relevant, test_grades, test_context, seed, MODEL_NAME, method
        )
        all_per_user.append(per_user)
        all_aggregate.append(aggregate)

    sensitivity_rows = []
    if RUN_CANDIDATE_SENSITIVITY:
        for pool_size in SENSITIVITY_POOL_SIZES:
            pools = final_pools if pool_size == PRIMARY_POOL_SIZE else build_candidate_pools(
                final_model, test_eval_users, trainval_seen, test_context["active_mask"], pool_size
            )
            for method in methods:
                rec_lists = {}
                for u_idx, (candidate_ids, candidate_scores) in pools.items():
                    weights = None if method in ("TopK", "Popularity") else selected["rerankers"][method]
                    rec_lists[u_idx] = rerank(
                        candidate_ids, candidate_scores, method, test_context, weights, K
                    )
                per_user, aggregate = evaluate_rec_lists(
                    rec_lists, test_relevant, test_grades, test_context, seed, MODEL_NAME, method
                )
                sensitivity_rows.append({
                    "pool_size": int(pool_size),
                    **aggregate,
                    "mean_ndcg": float(per_user["ndcg"].mean()),
                    "mean_ild": float(per_user["ild"].mean()),
                    "mean_list_tier_js": float(per_user["list_tier_js"].mean()),
                })

    result = {
        "seed": int(seed),
        "selected": selected,
        "base_history": base_history,
        "reranker_history": pd.concat(reranker_histories, ignore_index=True),
        "per_user": pd.concat(all_per_user, ignore_index=True),
        "aggregate": pd.DataFrame(all_aggregate),
        "sensitivity": pd.DataFrame(sensitivity_rows),
        "runtime_seconds": time.time() - start,
    }
    print(f"{MODEL_NAME} seed {seed} complete in {result['runtime_seconds'] / 60:.1f} minutes.")
    return result


seed_runs = [run_single_seed(seed) for seed in SEEDS]
print(f"All requested {MODEL_NAME} seeds completed.")


## 12. SVD aggregate results, confidence intervals, and paired significance tests

Paired tests compare JFDS with each baseline for the same model, seed, user, and candidate pool. The results table and tuning history are saved to the notebook's dedicated output folder.


In [ ]:
def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    if len(p_values) == 0:
        return p_values
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values), dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        candidate = min(1.0, (len(p_values) - rank) * p_values[idx])
        running_max = max(running_max, candidate)
        adjusted[idx] = running_max
    return adjusted


def paired_bootstrap_ci(differences: np.ndarray, n_boot: int, seed: int) -> tuple[float, float, float]:
    differences = np.asarray(differences, dtype=float)
    rng = np.random.default_rng(seed)
    sampled_means = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        sampled_means[b] = rng.choice(differences, size=len(differences), replace=True).mean()
    low, high = np.percentile(sampled_means, [2.5, 97.5])
    return float(differences.mean()), float(low), float(high)


def rank_biserial_from_differences(differences: np.ndarray) -> float:
    values = np.asarray(differences, dtype=float)
    values = values[~np.isclose(values, 0.0)]
    if len(values) == 0:
        return 0.0
    ranks = rankdata(np.abs(values), method="average")
    return float((ranks[values > 0].sum() - ranks[values < 0].sum()) / ranks.sum())


all_per_user_metrics = pd.concat([result["per_user"] for result in seed_runs], ignore_index=True)
all_aggregate_metrics = pd.concat([result["aggregate"] for result in seed_runs], ignore_index=True)
all_base_tuning = pd.concat([result["base_history"] for result in seed_runs], ignore_index=True)
all_reranker_tuning = pd.concat([result["reranker_history"] for result in seed_runs], ignore_index=True)
all_sensitivity = pd.concat([result["sensitivity"] for result in seed_runs], ignore_index=True)
selected_configurations = [result["selected"] for result in seed_runs]

metric_columns = ["precision", "recall", "ndcg", "hitrate", "ild", "list_tier_js", "self_information", "inverse_popularity"]
aggregate_columns = ["catalogue_coverage", "tail_catalogue_coverage", "exposure_gini", "tier_exposure_js", "tier_exposure_tv", "head_exposure_share", "mid_exposure_share", "tail_exposure_share", "distinct_recommended_items"]

seed_metric_means = (
    all_per_user_metrics.groupby(["seed", "base_model", "method"], as_index=False)[metric_columns].mean()
)
summary_per_user = (
    seed_metric_means.groupby(["base_model", "method"], as_index=False)[metric_columns].agg(["mean", "std"])
)
summary_per_user.columns = [
    "_".join(filter(None, col)).rstrip("_") for col in summary_per_user.columns.to_flat_index()
]

summary_aggregate = (
    all_aggregate_metrics.groupby(["base_model", "method"], as_index=False)[aggregate_columns].agg(["mean", "std"])
)
summary_aggregate.columns = [
    "_".join(filter(None, col)).rstrip("_") for col in summary_aggregate.columns.to_flat_index()
]

significance_rows = []
for (seed, base_model), group in all_per_user_metrics.groupby(["seed", "base_model"]):
    for baseline in ("TopK", "Popularity", "MMR", "FairOnly"):
        left = group[group["method"] == "JFDS"].set_index("user")
        right = group[group["method"] == baseline].set_index("user")
        common_users = left.index.intersection(right.index)
        for metric in metric_columns:
            diff = (
                left.loc[common_users, metric] - right.loc[common_users, metric]
            ).dropna().to_numpy(dtype=float)
            if len(diff) == 0:
                continue
            mean_delta, ci_low, ci_high = paired_bootstrap_ci(diff, N_BOOTSTRAP, seed + len(metric))
            try:
                p_value = (
                    float(wilcoxon(diff, zero_method="wilcox", alternative="two-sided").pvalue)
                    if not np.allclose(diff, 0.0) else 1.0
                )
            except ValueError:
                p_value = 1.0
            significance_rows.append({
                "seed": int(seed),
                "base_model": base_model,
                "comparison": f"JFDS - {baseline}",
                "metric": metric,
                "n_users": int(len(diff)),
                "mean_delta": mean_delta,
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "wilcoxon_p": p_value,
                "rank_biserial": rank_biserial_from_differences(diff),
            })

significance_tests = pd.DataFrame(significance_rows)
if not significance_tests.empty:
    significance_tests["holm_p"] = holm_adjust(significance_tests["wilcoxon_p"].to_numpy())
    significance_tests["significant_0_05"] = significance_tests["holm_p"] < 0.05

print("Per-user seed summary")
display(summary_per_user)
print("Aggregate exposure summary")
display(summary_aggregate)
print("Selected base parameters and re-ranker weights")
display(all_reranker_tuning.loc[
    all_reranker_tuning["selected"],
    ["seed", "base_model", "method", "alpha", "beta", "gamma",
     "validation_ndcg", "selection_status"]
])
print("Paired significance tests")
display(significance_tests.head(30))


## 13. Visual diagnostics

These figures are descriptive. They do not replace the validation protocol, paired confidence intervals, or Holm-adjusted tests.


In [ ]:
plot_per_user = seed_metric_means.groupby(["base_model", "method"], as_index=False)[metric_columns].mean()
plot_aggregate = all_aggregate_metrics.groupby(["base_model", "method"], as_index=False)[aggregate_columns].mean()
plot_df = plot_per_user.merge(plot_aggregate, on=["base_model", "method"], how="left")

for base_model, sub in plot_df.groupby("base_model"):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(sub["tier_exposure_js"], sub["ndcg"], s=80)
    for row in sub.itertuples(index=False):
        ax.annotate(row.method, (row.tier_exposure_js, row.ndcg), xytext=(5, 5), textcoords="offset points")
    ax.set_xlabel("Tier exposure JS divergence (lower is better)")
    ax.set_ylabel(f"NDCG@{K} (higher is better)")
    ax.set_title(f"{base_model}: relevance vs tier balance")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{base_model.lower()}_ndcg_vs_tier_balance.png", dpi=160)
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(sub["ild"], sub["ndcg"], s=80)
    for row in sub.itertuples(index=False):
        ax.annotate(row.method, (row.ild, row.ndcg), xytext=(5, 5), textcoords="offset points")
    ax.set_xlabel("Intra-list diversity (higher is better)")
    ax.set_ylabel(f"NDCG@{K} (higher is better)")
    ax.set_title(f"{base_model}: relevance vs genre diversity")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{base_model.lower()}_ndcg_vs_ild.png", dpi=160)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(sub))
    width = 0.25
    ax.bar(x - width, sub["head_exposure_share"], width, label="Head")
    ax.bar(x, sub["mid_exposure_share"], width, label="Mid")
    ax.bar(x + width, sub["tail_exposure_share"], width, label="Tail")
    ax.axhline(1/3, linestyle="--", linewidth=1, label="Policy target")
    ax.set_xticks(x, sub["method"], rotation=25, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Rank-discounted exposure share")
    ax.set_title(f"{base_model}: popularity-tier exposure")
    ax.legend()
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{base_model.lower()}_tier_exposure.png", dpi=160)
    plt.show()


## 14. SVD candidate-pool sensitivity and ablation view

Re-ranker weights are frozen after validation selection. Candidate-pool sizes are a test-time sensitivity analysis only. The table includes the primary tier-balance metric, relevance, diversity, and coverage.


In [ ]:
if RUN_CANDIDATE_SENSITIVITY:
    sensitivity_summary = (
        all_sensitivity.groupby(["base_model", "method", "pool_size"], as_index=False)
        .agg(
            mean_ndcg=("mean_ndcg", "mean"),
            sd_ndcg=("mean_ndcg", "std"),
            mean_ild=("mean_ild", "mean"),
            mean_list_tier_js=("mean_list_tier_js", "mean"),
            mean_tier_exposure_js=("tier_exposure_js", "mean"),
            mean_coverage=("catalogue_coverage", "mean"),
        )
    )
    display(sensitivity_summary)
else:
    sensitivity_summary = pd.DataFrame()
    print("Candidate-pool sensitivity disabled.")

ablation_table = plot_df[
    ["base_model", "method", "ndcg", "ild", "list_tier_js", "tier_exposure_js",
     "tail_exposure_share", "catalogue_coverage"]
].sort_values(["base_model", "method"])
display(ablation_table)


## 15. Save reproducible SVD outputs

No outcomes are fabricated. Executing the notebook writes metrics, validation tuning histories, selected weights, plots, and run metadata to the dedicated results folder.


In [ ]:
def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value

all_per_user_metrics.to_csv(OUTPUT_DIR / "per_user_metrics.csv", index=False)
all_aggregate_metrics.to_csv(OUTPUT_DIR / "aggregate_metrics_by_seed.csv", index=False)
summary_per_user.to_csv(OUTPUT_DIR / "per_user_summary_across_seeds.csv", index=False)
summary_aggregate.to_csv(OUTPUT_DIR / "aggregate_exposure_summary_across_seeds.csv", index=False)
significance_tests.to_csv(OUTPUT_DIR / "significance_tests.csv", index=False)
all_base_tuning.to_csv(OUTPUT_DIR / "base_model_tuning_history.csv", index=False)
all_reranker_tuning.to_csv(OUTPUT_DIR / "reranker_tuning_history.csv", index=False)
warm_start_report.to_csv(OUTPUT_DIR / "warm_start_exclusions.csv", index=False)
dropped_users_df.to_csv(OUTPUT_DIR / "dropped_users.csv", index=False)
if not all_sensitivity.empty:
    all_sensitivity.to_csv(OUTPUT_DIR / "candidate_pool_sensitivity_by_seed.csv", index=False)
    sensitivity_summary.to_csv(OUTPUT_DIR / "candidate_pool_sensitivity_summary.csv", index=False)

with open(OUTPUT_DIR / "selected_hyperparameters_and_weights.json", "w", encoding="utf-8") as handle:
    json.dump(json_ready(selected_configurations), handle, indent=2)

run_metadata = {
    "base_model": MODEL_NAME,
    "jfds_equation_version": "v2",
    "seeds": SEEDS,
    "split": {"train": TRAIN_FRACTION, "validation": VAL_FRACTION, "test": TEST_FRACTION},
    "positive_threshold": POSITIVE_THRESHOLD,
    "k": K,
    "primary_pool_size": PRIMARY_POOL_SIZE,
    "candidate_pool_sensitivity": SENSITIVITY_POOL_SIZES,
    "tune_user_limit": TUNE_USER_LIMIT,
    "weight_step": WEIGHT_STEP,
    "min_active_weight": MIN_ACTIVE_WEIGHT,
    "selection_policy": SELECTION_POLICY,
    "warm_start_report": warm_start_report.to_dict(orient="records"),
}
with open(OUTPUT_DIR / "run_metadata.json", "w", encoding="utf-8") as handle:
    json.dump(json_ready(run_metadata), handle, indent=2)

print("Saved files:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(" -", path)


## 16. Interpretation template and limitations — SVD

After execution, report:

1. The validation-selected JFDS weights for each seed and `selection_status = objective_only_non_degenerate`.
2. Whether JFDS improved tier balance and genre diversity versus the relevant baselines.
3. The test NDCG@10 result as an outcome, not as a selection target.
4. Paired confidence intervals, Holm-adjusted tests, seed stability, and candidate-pool sensitivity.
5. The full validation objective surface from `reranker_tuning_history.csv`, including the selected non-degenerate weight set.

### Limitations

- This is an offline warm-start MovieLens experiment, not evidence of online satisfaction.
- Version 2's equal tier target is a declared policy target, not a universal formal fairness definition.
- The candidate generator can limit what any re-ranker can recover.
- Do not infer demographic or provider-side fairness claims from this experiment.
